# Stereo camera calibration

Calibrates the pair of INNO-MAKER UVC cameras: intrinsics (K, distortion) for each camera, then the
extrinsics between them (R, T), plus rectification maps for stereo.

Workflow
1. Print a chessboard (default 9x6 inner corners), tape it to something **flat and rigid**, measure one square with calipers and set `SQUARE_M`.
2. Run the config + camera probe cells, then the capture cell. Show the board to both cameras and press **SPACE** to save a pair (only saved if the board is found in both). Aim for 20-30 pairs: vary distance, tilt (+-30 deg), and cover the corners of each image. **q** quits.
3. Run the rest. Results go to `calib/stereo_calib.json` (same OpenCV convention as `gen_pof3d.py`: `X_cam1 = R @ X_cam0 + T`).

The cameras are plain UVC devices with no hardware sync, so hold the board **still** when you press SPACE.
Lock focus/exposure (AMCap or the cell below) so intrinsics don't drift between calibration and use.

In [1]:
import cv2
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- config -----------------------------------------------------------------
CAM_IDS = (0, 1)            # OpenCV indices of the left / right camera (see probe cell)
WIDTH, HEIGHT = 1280, 720   # must be a mode the camera supports (check in AMCap)
FPS = 30
FOURCC = "MJPG"             # MJPG allows higher res/fps over USB2 than YUYV
BACKEND = cv2.CAP_DSHOW     # DirectShow on Windows; MSMF also works but is slower to open

BOARD = (9, 6)              # inner corners (cols, rows) of the chessboard
SQUARE_M = 0.025            # square edge length [m] -- measure your print!

OUT = Path("calib")
IMG_DIR = OUT / "images"
print(cv2.__version__, Path.cwd())

5.0.0 c:\Users\alagi\shape-sensing\sw


In [2]:
# probe which indices are cameras (laptop webcams usually take 0)
for i in range(6):
    cap = cv2.VideoCapture(i, BACKEND)
    if cap.isOpened():
        ok, f = cap.read()
        print(i, "ok" if ok else "opened, no frame", None if f is None else f.shape)
    cap.release()

0 ok (480, 640, 3)


In [4]:
def open_cam(idx):
    cap = cv2.VideoCapture(idx, BACKEND)
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*FOURCC))
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)
    cap.set(cv2.CAP_PROP_FPS, FPS)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    # lock focus; with DSHOW, CAP_PROP_SETTINGS opens the driver dialog for exposure/WB etc.
    cap.set(cv2.CAP_PROP_AUTOFOCUS, 0)
    w, h = cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    assert cap.isOpened(), f"camera {idx} did not open"
    print(f"cam {idx}: {w:.0f}x{h:.0f} @ {cap.get(cv2.CAP_PROP_FPS):.0f} fps")
    return cap

def read_pair(caps):
    # grab both first, then decode, to keep the two exposures as close in time as possible
    for c in caps:
        c.grab()
    return [c.retrieve()[1] for c in caps]

CB_FLAGS = cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE | cv2.CALIB_CB_FAST_CHECK

def find_board(img, fast=True):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ok, c = cv2.findChessboardCorners(gray, BOARD, CB_FLAGS if fast else CB_FLAGS & ~cv2.CALIB_CB_FAST_CHECK)
    if ok:
        c = cv2.cornerSubPix(gray, c, (11, 11), (-1, -1),
                             (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 50, 1e-3))
    return ok, c

In [5]:
# live capture: SPACE = save pair (only if board seen in both), s = driver settings dialog, q = quit
IMG_DIR.mkdir(parents=True, exist_ok=True)
n = len(list(IMG_DIR.glob("left_*.png")))
caps = [open_cam(i) for i in CAM_IDS]
coverage = [np.zeros((HEIGHT, WIDTH), np.uint8) for _ in caps]   # where corners have been collected
try:
    while True:
        frames = read_pair(caps)
        if any(f is None for f in frames):
            print("dropped frame"); continue
        found = [find_board(f) for f in frames]
        views = []
        for f, (ok, c), cov in zip(frames, found, coverage):
            v = f.copy()
            v[cov > 0] = (0.6 * v[cov > 0] + [0, 100, 0]).astype(np.uint8)   # green tint = covered
            cv2.drawChessboardCorners(v, BOARD, c, ok)
            views.append(v)
        vis = np.hstack(views)
        cv2.putText(vis, f"pairs: {n}   SPACE save | s settings | q quit", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
        scale = 1600 / vis.shape[1]
        cv2.imshow("stereo calib", cv2.resize(vis, None, fx=scale, fy=scale))
        k = cv2.waitKey(1) & 0xFF
        if k == ord("q"):
            break
        if k == ord("s"):
            for c in caps:
                c.set(cv2.CAP_PROP_SETTINGS, 1)
        if k == ord(" "):
            if all(ok for ok, _ in found):
                cv2.imwrite(str(IMG_DIR / f"left_{n:03d}.png"), frames[0])
                cv2.imwrite(str(IMG_DIR / f"right_{n:03d}.png"), frames[1])
                for (_, c), cov in zip(found, coverage):
                    cv2.fillConvexPoly(cov, cv2.convexHull(c.reshape(-1, 2).astype(np.int32)), 1)
                n += 1
                print("saved pair", n)
            else:
                print("board not found in both views, not saved")
finally:
    for c in caps:
        c.release()
    cv2.destroyAllWindows()

cam 0: 1280x720 @ 30 fps


AssertionError: camera 1 did not open

In [ ]:
# detect corners in all saved pairs
objp = np.zeros((BOARD[0] * BOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:BOARD[0], 0:BOARD[1]].T.reshape(-1, 2) * SQUARE_M

names, obj_pts, img_pts = [], [], ([], [])
for lp in sorted(IMG_DIR.glob("left_*.png")):
    rp = IMG_DIR / lp.name.replace("left_", "right_")
    imgs = [cv2.imread(str(lp)), cv2.imread(str(rp))]
    found = [find_board(im, fast=False) for im in imgs]
    if not all(ok for ok, _ in found):
        print("skip", lp.name); continue
    names.append(lp.stem.split("_")[1])
    obj_pts.append(objp)
    for lst, (_, c) in zip(img_pts, found):
        lst.append(c)
image_size = imgs[0].shape[1::-1]   # (w, h)
print(f"{len(names)} usable pairs, image size {image_size}")

In [ ]:
# per-camera intrinsics, then drop views with large reprojection error and refit
def calib_mono(obj, img):
    rms, K, D, rv, tv = cv2.calibrateCamera(obj, img, image_size, None, None)
    err = [np.sqrt(np.mean(np.sum((im.reshape(-1, 2) - cv2.projectPoints(o, r, t, K, D)[0].reshape(-1, 2))**2, 1)))
           for o, im, r, t in zip(obj, img, rv, tv)]
    return rms, K, D, np.array(err)

res = [calib_mono(obj_pts, ip) for ip in img_pts]
worst = np.maximum(res[0][3], res[1][3])
keep = worst < max(1.0, 3 * np.median(worst))   # px
if not keep.all():
    print("dropping views:", [names[i] for i in np.where(~keep)[0]])
    obj_pts = [o for o, k in zip(obj_pts, keep) if k]
    img_pts = tuple([c for c, k in zip(ip, keep) if k] for ip in img_pts)
    names = [nm for nm, k in zip(names, keep) if k]
    res = [calib_mono(obj_pts, ip) for ip in img_pts]

(rms0, K0, D0, e0), (rms1, K1, D1, e1) = res
for nm, rms, K, D in [("left", rms0, K0, D0), ("right", rms1, K1, D1)]:
    hfov = np.degrees(2 * np.arctan(image_size[0] / (2 * K[0, 0])))
    print(f"{nm}: rms {rms:.3f} px  fx={K[0,0]:.1f} fy={K[1,1]:.1f} cx={K[0,2]:.1f} cy={K[1,2]:.1f}  "
          f"hfov={hfov:.1f} deg  D={np.round(D.ravel(), 4)}")

fig, ax = plt.subplots(figsize=(10, 3))
x = np.arange(len(names))
ax.bar(x - 0.2, e0, 0.4, label="left"); ax.bar(x + 0.2, e1, 0.4, label="right")
ax.set_xticks(x, names, rotation=90); ax.set_ylabel("reproj. error [px]"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# stereo extrinsics with intrinsics held fixed: X_right = R @ X_left + T
rms_st, *_, R, T, E, F = cv2.stereoCalibrate(
    obj_pts, img_pts[0], img_pts[1], K0, D0, K1, D1, image_size,
    flags=cv2.CALIB_FIX_INTRINSIC,
    criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 1e-6))

rvec = cv2.Rodrigues(R)[0].ravel()
print(f"stereo rms {rms_st:.3f} px")
print(f"baseline {np.linalg.norm(T)*1e3:.2f} mm   T = {np.round(T.ravel()*1e3, 2)} mm")
print(f"relative rotation {np.degrees(np.linalg.norm(rvec)):.2f} deg   rvec = {np.round(np.degrees(rvec), 2)} deg")

In [ ]:
# rectification + save
R0, R1, P0, P1, Q, roi0, roi1 = cv2.stereoRectify(K0, D0, K1, D1, image_size, R, T, alpha=0)
maps = [cv2.initUndistortRectifyMap(K, D, Rr, P, image_size, cv2.CV_16SC2)
        for K, D, Rr, P in [(K0, D0, R0, P0), (K1, D1, R1, P1)]]

calib = dict(
    image_size=list(image_size), board=list(BOARD), square_m=SQUARE_M, n_pairs=len(names),
    cam_ids=list(CAM_IDS), rms_px=dict(left=rms0, right=rms1, stereo=rms_st),
    K0=K0, D0=D0.ravel(), K1=K1, D1=D1.ravel(), R=R, T=T.ravel(), E=E, F=F,
    R0=R0, R1=R1, P0=P0, P1=P1, Q=Q, baseline_m=float(np.linalg.norm(T)),
)
calib = {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in calib.items()}
(OUT / "stereo_calib.json").write_text(json.dumps(calib, indent=1))
print("saved", OUT / "stereo_calib.json")

In [ ]:
# check: rectified pair, matching points should lie on the same horizontal line
lp, rp = IMG_DIR / f"left_{names[0]}.png", IMG_DIR / f"right_{names[0]}.png"
rect = [cv2.remap(cv2.imread(str(p)), m1, m2, cv2.INTER_LINEAR) for p, (m1, m2) in zip((lp, rp), maps)]

# epipolar error: rectified corner y-coordinates should match between views
dy = []
for c0, c1 in zip(*img_pts):
    u0 = cv2.undistortPoints(c0, K0, D0, R=R0, P=P0)
    u1 = cv2.undistortPoints(c1, K1, D1, R=R1, P=P1)
    dy.append(u0[:, 0, 1] - u1[:, 0, 1])
dy = np.concatenate(dy)
print(f"rectified |dy|: mean {np.abs(dy).mean():.3f} px, max {np.abs(dy).max():.3f} px")

vis = np.hstack(rect)
for y in range(0, vis.shape[0], 40):
    cv2.line(vis, (0, y), (vis.shape[1], y), (0, 255, 0), 1)
plt.figure(figsize=(14, 5)); plt.imshow(vis[..., ::-1]); plt.axis("off"); plt.show()

In [ ]:
# load later with:
def load_calib(path=OUT / "stereo_calib.json"):
    c = json.loads(Path(path).read_text())
    return {k: np.array(v) if isinstance(v, list) else v for k, v in c.items()}

c = load_calib()
# triangulate matched pixels (N,2) from left/right raw images -> (N,3) in left-camera frame [m]
def triangulate(uv0, uv1, c=c):
    p0 = cv2.undistortPoints(uv0.reshape(-1, 1, 2).astype(np.float64), c["K0"], c["D0"])
    p1 = cv2.undistortPoints(uv1.reshape(-1, 1, 2).astype(np.float64), c["K1"], c["D1"])
    P0 = np.hstack([np.eye(3), np.zeros((3, 1))])
    P1 = np.hstack([c["R"], c["T"].reshape(3, 1)])
    X = cv2.triangulatePoints(P0, P1, p0.reshape(-1, 2).T, p1.reshape(-1, 2).T)
    return (X[:3] / X[3]).T

# sanity: board corners of first pair -> neighbouring-corner spacing should equal SQUARE_M
X = triangulate(img_pts[0][0].reshape(-1, 2), img_pts[1][0].reshape(-1, 2))
d = np.linalg.norm(np.diff(X.reshape(BOARD[1], BOARD[0], 3), axis=1), axis=-1)
print(f"corner spacing {d.mean()*1e3:.2f} +- {d.std()*1e3:.2f} mm (expected {SQUARE_M*1e3:.1f}), "
      f"board distance {np.linalg.norm(X.mean(0))*1e3:.0f} mm")